In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import yfinance as yf
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from scipy import stats

In [ ]:
tickers = ['^BVSP', 'BRL=X', '^VIX', 'CL=F', 'GC=F'] # ibov, cambio, vix, petroleo, ouro
start_date = '2010-01-01' 
end_date =  '2025-01-01'
print(start_date, end_date)

def pegarCotacoes(tickers, start_date, end_date):
    df = yf.download(tickers, start_date, end_date, auto_adjust=False)
    df = df['Adj Close']
    return df

df_cotacoes = pegarCotacoes(tickers, start_date, end_date)
df_cotacoes = df_cotacoes.ffill()
df_cotacoes = df_cotacoes.ffill().dropna()
display(df_cotacoes)

2010-01-01 2025-01-01


[*********************100%***********************]  5 of 5 completed


Ticker,BRL=X,CL=F,GC=F,^BVSP,^VIX
Date,,,,,
2010-01-04,1.7190,81.510002,1117.699951,70045.0,20.040001
2010-01-05,1.7370,81.769997,1118.099976,70240.0,19.350000
2010-01-06,1.7315,83.180000,1135.900024,70729.0,19.160000
2010-01-07,1.7389,82.660004,1133.099976,70451.0,19.059999
2010-01-08,1.7320,82.750000,1138.199951,70263.0,18.129999
...,...,...,...,...,...
2024-12-25,6.1756,70.099998,2620.000000,120767.0,14.270000
2024-12-26,6.1828,69.620003,2638.800049,121078.0,14.730000
2024-12-27,6.1485,70.599998,2617.199951,120269.0,15.950000


In [22]:
# retorno de logs diarios
df_returns = np.log(df_cotacoes/df_cotacoes.shift(1)).dropna()
df_returns.columns = ['CAMBIO', 'PETROLEO', 'OURO', 'IBOV','VIX']

# features
df_features = pd.DataFrame(index= df_returns.index)

# retorno IBOV 
df_features['ret_ibov'] = df_returns['IBOV']

# volatilidade realizada (janela 21 dias)
df_features['vol_21'] = df_returns['IBOV'].rolling(21).std() * np.sqrt(252)

# zscore do retorno (janela 63 dias)
rolling_mean = df_returns['IBOV'].rolling(63).mean()
rolling_std = df_returns['IBOV'].rolling(63).std()
df_features['zscore_63'] = (df_returns['IBOV'] - rolling_mean) / rolling_std 

# variaveis macro
df_features['ret_cambio'] = df_returns['CAMBIO']
df_features['ret_vix'] = df_returns['VIX']
df_features['ret_petroleo'] = df_returns['PETROLEO']
df_features['ret_ouro'] = df_returns['OURO']

# momentum (retorno acumulado 21 dias)
df_features['mom_21'] = df_returns['IBOV'].rolling(21).sum()

df_features.dropna(inplace=True)

print(df_features.shape)
print(df_features.describe().round(4))

(3847, 8)
        ret_ibov     vol_21  zscore_63  ret_cambio    ret_vix  ret_petroleo  \
count  3847.0000  3847.0000  3847.0000   3847.0000  3847.0000     3847.0000   
mean      0.0001     0.2074    -0.0193      0.0003    -0.0000        0.0001   
std       0.0146     0.1022     1.0117      0.0107     0.0766        0.0254   
min      -0.1599     0.0666    -5.5667     -0.0638    -0.3506       -0.2822   
25%      -0.0073     0.1538    -0.6176     -0.0053    -0.0414       -0.0109   
50%       0.0000     0.1890    -0.0092      0.0001    -0.0035        0.0001   
75%       0.0080     0.2366     0.6097      0.0059     0.0329        0.0119   
max       0.1302     1.2930     3.6457      0.0725     0.7682        0.3196   

        ret_ouro     mom_21  
count  3847.0000  3847.0000  
mean      0.0002     0.0032  
std       0.0099     0.0660  
min      -0.0982    -0.5813  
25%      -0.0043    -0.0329  
50%       0.0001     0.0039  
75%       0.0053     0.0444  
max       0.0578     0.2331  


In [ ]:

from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression

# ret_ibov como série principal
modelos_msar = MarkovAutoregression(df_features['ret_ibov'], k_regimes=3, order=1, switching_ar=True, switching_variance=True)
resultado = modelos_msar.fit(search_reps=20, search_scale=True, disp=False)

print(resultado.summary())

c:\Users\Usuario\Documents\quantai\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Usuario\Documents\quantai\.venv\Lib\site-packages\statsmodels\tsa\regime_switching\markov_switching.py:1292: EstimationWarning: Invalid regime transition probabilities estimated in EM iteration; probabilities have been re-scaled to continue estimation.
  warnings.warn('Invalid regime transition probabilities'
c:\Users\Usuario\Documents\quantai\.venv\Lib\site-packages\statsmodels\tsa\regime_switching\markov_switching.py:1292: EstimationWarning: Invalid regime transition probabilities estimated in EM iteration; probabilities have been re-scaled to continue estimation.
  warnings.warn('Invalid regime transition probabilities'
c:\Users\Usuario\Documents\quantai\.venv\Lib\site-packages\statsmodels\tsa\regime_switching

                         Markov Switching Model Results                         
Dep. Variable:                 ret_ibov   No. Observations:                 3846
Model:             MarkovAutoregression   Log Likelihood               11298.127
Date:                  Wed, 22 Jul 2026   AIC                         -22566.254
Time:                          16:56:25   BIC                         -22472.432
Sample:                               0   HQIC                        -22532.930
                                 - 3846                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0002      0.000     -0.429      0.668      -0.001       0.001
sigma2         0.0003   1.72e-05    